# ❤️ New RVC — Training Notebook

**Dedicated high-speed training pipeline** — bypasses Gradio UI overhead for maximum GPU utilization.

---

### 🚀 Why use this notebook?
- **Direct function calls** — no Gradio event loop, websocket, or UI rendering overhead
- **Full control** over hyperparameters (learning rate, decay, seed, vocoder)
- **Real-time monitoring** with TensorBoard integration
- **Auto-saves best model** using Smart Auto-Tuning

### 📝 Quick Start
1. Run cells 1-2 to set up environment
2. Upload your dataset (Cell 3)
3. Run preprocessing → extraction → training (Cells 4-6)
4. Download your trained model (Cell 7)

In [ ]:
# @title 1️⃣ Clone & Install
# @markdown > This takes ~1-2 min on Colab T4
import os

if os.path.exists('/content/NEWRVC'):
    !rm -rf /content/NEWRVC
!git clone https://github.com/DDME36/NEWRVC.git /content/NEWRVC
%cd /content/NEWRVC

!apt install -y python3-dev unzip -qq
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
os.environ['URVC_CONSOLE_LOG_LEVEL'] = 'INFO'

# Remove stale lock file
if os.path.exists('uv.lock'):
    os.remove('uv.lock')

# Initialize core engine + download prerequisites
!uv run --prerelease if-necessary-or-explicit ./src/ultimate_rvc/core/main.py
!uv add --prerelease if-necessary-or-explicit matplotlib-inline==0.1.7
print('\n\u2705 Setup complete!')

In [ ]:
# @title 2️⃣ Load Training Modules
import sys
sys.path.insert(0, '/content/NEWRVC/src')

from ultimate_rvc.core.train.prepare import populate_dataset, preprocess_dataset
from ultimate_rvc.core.train.extract import extract_features
from ultimate_rvc.core.train.train import run_training
from ultimate_rvc.typing_extra import (
    TrainingSampleRate,
    AudioSplitMethod,
    F0Method,
    EmbedderModel,
    Vocoder,
    IndexAlgorithm,
    PretrainedType,
    PrecisionType,
)
import time
print('\u2705 Modules loaded!')

In [ ]:
# @title 3️⃣ Upload & Create Dataset
# @markdown ### Dataset Settings
# @markdown Upload audio files (.wav, .mp3, .flac) or provide a path
model_name = 'My_Model'  # @param {type:"string"}
dataset_name = 'My_Dataset'  # @param {type:"string"}

# @markdown ---
# @markdown ### Option A: Upload files directly
use_upload = True  # @param {type:"boolean"}

# @markdown ### Option B: Use existing path (Google Drive, etc.)
dataset_path = '/content/drive/MyDrive/dataset'  # @param {type:"string"}

if use_upload:
    from google.colab import files
    print('\ud83d\udcc1 Upload your audio files (wav/mp3/flac):')
    uploaded = files.upload()
    
    # Save to temp dir
    import shutil
    from pathlib import Path
    upload_dir = Path('/content/uploaded_audio')
    upload_dir.mkdir(exist_ok=True)
    for fname, data in uploaded.items():
        (upload_dir / fname).write_bytes(data)
    
    audio_files = list(upload_dir.glob('*'))
    print(f'\n\ud83c\udfa4 Uploaded {len(audio_files)} file(s)')
    
    # Populate dataset
    ds_path = populate_dataset(dataset_name, [str(f) for f in audio_files])
    dataset_path = str(ds_path)
    print(f'\u2705 Dataset created at: {dataset_path}')
else:
    print(f'\ud83d\udcc2 Using existing dataset at: {dataset_path}')

In [ ]:
# @title 4️⃣ Preprocess Dataset
# @markdown ### Preprocessing Settings
sample_rate = '40000'  # @param ['32000', '40000', '48000']
split_method = 'Automatic'  # @param ['Automatic', 'Simple', 'Skip']
filter_audio = True  # @param {type:"boolean"}
clean_audio = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.1}

sr_map = {'32000': TrainingSampleRate.HZ_32K, '40000': TrainingSampleRate.HZ_40K, '48000': TrainingSampleRate.HZ_48K}
split_map = {'Automatic': AudioSplitMethod.AUTOMATIC, 'Simple': AudioSplitMethod.SIMPLE, 'Skip': AudioSplitMethod.SKIP}

print(f'\ud83d\udd27 Preprocessing with SR={sample_rate}, split={split_method}...')
t0 = time.time()

preprocess_dataset(
    model_name=model_name,
    dataset=dataset_path,
    sample_rate=sr_map[sample_rate],
    split_method=split_map[split_method],
    filter_audio=filter_audio,
    clean_audio=clean_audio,
    clean_strength=clean_strength,
)

elapsed = time.time() - t0
print(f'\u2705 Preprocessing complete in {elapsed:.1f}s')

In [ ]:
# @title 5️⃣ Extract Features
# @markdown ### Feature Extraction Settings
f0_method = 'rmvpe'  # @param ['rmvpe', 'crepe', 'fcpe', 'hybrid[rmvpe+fcpe]']
embedder_model = 'contentvec'  # @param ['contentvec', 'japanese-hubert-base', 'chinese-hubert-large', 'spin']

f0_map = {
    'rmvpe': F0Method.RMVPE, 'crepe': F0Method.CREPE,
    'fcpe': F0Method.FCPE, 'hybrid[rmvpe+fcpe]': F0Method.HYBRID_RMVPE_FCPE
}
emb_map = {
    'contentvec': EmbedderModel.CONTENTVEC,
    'japanese-hubert-base': EmbedderModel.JAPANESE_HUBERT_BASE,
    'chinese-hubert-large': EmbedderModel.CHINESE_HUBERT_LARGE,
    'spin': EmbedderModel.SPIN,
}

print(f'\ud83d\udd0d Extracting features (F0={f0_method}, Embedder={embedder_model})...')
t0 = time.time()

extract_features(
    model_name=model_name,
    f0_method=f0_map[f0_method],
    embedder_model=emb_map[embedder_model],
)

elapsed = time.time() - t0
print(f'\u2705 Feature extraction complete in {elapsed:.1f}s')

In [ ]:
# @title 6️⃣ Train Voice Model
# @markdown ### Core Training Settings
num_epochs = 500  # @param {type:"slider", min:50, max:2000, step:50}
batch_size = 8  # @param {type:"slider", min:1, max:32, step:1}
vocoder = 'HiFi-GAN'  # @param ['HiFi-GAN', 'MRF HiFi-GAN', 'RefineGAN', 'RingFormer_v1', 'RingFormer_v2', 'APEX-GAN']
pretrained = 'Default'  # @param ['Default', 'None']
precision = 'fp16'  # @param ['fp32', 'fp16', 'bf16']

# @markdown ---
# @markdown ### Advanced Hyperparameters
learning_rate = 0.0001  # @param {type:"number"}
lr_decay = 0.999875  # @param {type:"number"}
seed = 1234  # @param {type:"integer"}

# @markdown ---
# @markdown ### Safety & Saving
detect_overtraining = True  # @param {type:"boolean"}
overtraining_threshold = 50  # @param {type:"slider", min:10, max:200, step:10}
save_interval = 10  # @param {type:"slider", min:5, max:50, step:5}
save_all_weights = False  # @param {type:"boolean"}
reduce_memory = False  # @param {type:"boolean"}

vocoder_map = {
    'HiFi-GAN': Vocoder.HIFI_GAN, 'MRF HiFi-GAN': Vocoder.MRF_HIFI_GAN,
    'RefineGAN': Vocoder.REFINE_GAN, 'RingFormer_v1': Vocoder.RINGFORMER_V1,
    'RingFormer_v2': Vocoder.RINGFORMER_V2, 'APEX-GAN': Vocoder.APEX_GAN,
}
pretrained_map = {'Default': PretrainedType.DEFAULT, 'None': PretrainedType.NONE}
precision_map = {'fp32': PrecisionType.FP32, 'fp16': PrecisionType.FP16, 'bf16': PrecisionType.BF16}

print(f'\ud83c\udfaf Training "{model_name}" — {num_epochs} epochs, batch={batch_size}, {vocoder}, {precision}')
print(f'    LR={learning_rate}, Decay={lr_decay}, Seed={seed}')
print(f'    Overtraining detection: {"ON" if detect_overtraining else "OFF"} (threshold={overtraining_threshold})')
print('=' * 60)
t0 = time.time()

result = run_training(
    model_name=model_name,
    num_epochs=num_epochs,
    batch_size=batch_size,
    detect_overtraining=detect_overtraining,
    overtraining_threshold=overtraining_threshold,
    vocoder=vocoder_map[vocoder],
    index_algorithm=IndexAlgorithm.AUTO,
    pretrained_type=pretrained_map[pretrained],
    save_interval=save_interval,
    save_all_weights=save_all_weights,
    precision=precision_map[precision],
    reduce_memory_usage=reduce_memory,
    learning_rate=learning_rate,
    lr_decay=lr_decay,
    seed=seed,
)

elapsed = time.time() - t0
print('\n' + '=' * 60)
if result:
    print(f'\u2705 Training complete in {elapsed/60:.1f} minutes!')
    print(f'   Model: {result[0]}')
    print(f'   Index: {result[1]}')
else:
    print(f'\u26a0\ufe0f Training finished in {elapsed/60:.1f} min but no best model was saved.')

In [ ]:
# @title 📊 TensorBoard (Optional)
# @markdown Launch TensorBoard to monitor training in real-time.
# @markdown Run this cell **before** or **during** training.

import os
log_dir = f'/content/NEWRVC/models/training/{model_name}/logs'
if os.path.exists(log_dir):
    %load_ext tensorboard
    %tensorboard --logdir {log_dir}
else:
    print(f'\u26a0\ufe0f No logs found at {log_dir}')
    print('Start training first, then re-run this cell.')

In [ ]:
# @title 7️⃣ Download Trained Model
# @markdown Downloads the .pth weights + .index as a zip file.

import zipfile
from pathlib import Path

if result:
    zip_path = f'/content/{model_name}_trained.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fpath in result:
            zf.write(fpath, Path(fpath).name)
    
    print(f'\ud83d\udce6 Model packaged: {zip_path}')
    
    from google.colab import files
    files.download(zip_path)
else:
    print('\u274c No model to download. Run training first.')

---

## 📝 Training Tips

| Setting | Speech | Singing | Cross-Gender |
|---------|--------|---------|-------------|
| Dataset | 10-30 min | 30-60 min | 30-60 min |
| Epochs | 200-400 | 400-800 | 500-800 |
| Batch Size | 8-12 | 6-8 | 6-8 |
| Vocoder | HiFi-GAN | RingFormer_v2 | RefineGAN |
| LR | 1e-4 | 5e-5 | 5e-5 |
| Precision | FP16 | BF16/FP32 | FP16 |

### ⚠️ Important Notes
- **RingFormer/APEX-GAN** don't have pretrained models yet — use `Pretrained=None` and train longer (800+ epochs)
- **Overtraining detection ON** is always recommended (default threshold=50 works for most cases)
- **FP16** saves ~30% VRAM and is 1.5-2x faster on T4/A100
- Monitor **TensorBoard** — if val loss stops decreasing, training will auto-stop